# 07 · Ajuste de Hiperparámetros

**Etapa 4-5 (semana 12)** — "Ajuste de hiperparámetros y evaluación de modelos".

Busca la mejor configuración para cada uno de los 3 modelos, probando varias combinaciones
sobre una muestra del dataset (rápido) y evaluando F1 en el set de validación. Al final
te da la tabla comparativa (para el informe) y la configuración recomendada para usar en
`00_pipeline_modelado.ipynb` en la corrida final con el dataset completo.

**Entrada:** `train.parquet`, `val.parquet` (generados por `01_exploracion_dataset.ipynb`)
**Salidas:** `resultados/tablas/07_busqueda_hiperparametros.csv`,
`resultados/figuras/09_busqueda_hiperparametros.png`


In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score

USUARIO = os.path.basename(os.path.expanduser("~"))
DATOS_BASE_DIR = os.path.join("/data", USUARIO, "deteccion_incendios")
BASE_DIR = os.path.join(os.path.expanduser("~"), "deteccion_incendios", "modelado")

DIR_DATOS = os.path.join(DATOS_BASE_DIR, "datos")
DIR_TABLAS = os.path.join(BASE_DIR, "resultados/tablas")
DIR_FIGURAS = os.path.join(BASE_DIR, "resultados/figuras")
os.makedirs(DIR_TABLAS, exist_ok=True)
os.makedirs(DIR_FIGURAS, exist_ok=True)

TARGET = "es_falsa_alarma"
SEMILLA = 42
FRACCION_BUSQUEDA = 0.15  # para que la busqueda sea rapida; el mejor config se aplica luego al dataset completo


## 1. Carga de datos

In [ ]:
train_pd = pd.read_parquet(os.path.join(DIR_DATOS, "train.parquet")).sample(frac=FRACCION_BUSQUEDA, random_state=SEMILLA)
val_pd   = pd.read_parquet(os.path.join(DIR_DATOS, "val.parquet")).sample(frac=FRACCION_BUSQUEDA, random_state=SEMILLA)

X_train, y_train = train_pd.drop(columns=[TARGET]), train_pd[TARGET]
X_val,   y_val   = val_pd.drop(columns=[TARGET]),   val_pd[TARGET]
print(f"Busqueda con {len(y_train):,} filas de train, {len(y_val):,} de val")

resultados_busqueda = []


## 2. Random Forest — grid de `n_estimators` x `max_depth`

In [ ]:
try:
    import cudf
    from cuml.ensemble import RandomForestClassifier as cuRF
    USA_GPU_RF = True
except ImportError:
    from sklearn.ensemble import RandomForestClassifier as skRF
    USA_GPU_RF = False

print("Random Forest backend:", "cuML/GPU" if USA_GPU_RF else "sklearn/CPU")

grid_rf = [
    {"n_estimators": 100, "max_depth": 8},
    {"n_estimators": 100, "max_depth": 16},
    {"n_estimators": 300, "max_depth": 16},
    {"n_estimators": 300, "max_depth": 24},
]

if USA_GPU_RF:
    Xtr_gpu = cudf.DataFrame.from_pandas(X_train.astype("float32"))
    ytr_gpu = cudf.Series(y_train.values.astype("int32"))
    Xval_gpu = cudf.DataFrame.from_pandas(X_val.astype("float32"))

for params in grid_rf:
    t0 = time.perf_counter()
    if USA_GPU_RF:
        modelo = cuRF(n_streams=4, random_state=SEMILLA, **params)
        modelo.fit(Xtr_gpu, ytr_gpu)
        pred = modelo.predict(Xval_gpu)
        pred = pred.to_numpy() if hasattr(pred, "to_numpy") else pred
    else:
        modelo = skRF(n_jobs=-1, class_weight="balanced", random_state=SEMILLA, **params)
        modelo.fit(X_train, y_train)
        pred = modelo.predict(X_val)
    tiempo = time.perf_counter() - t0
    f1 = f1_score(y_val, pred)
    print(f"RF {params} -> F1={f1:.4f}  ({tiempo:.1f}s)")
    resultados_busqueda.append({"modelo": "Random Forest", "config": str(params), "f1": f1, "tiempo_s": tiempo, **params})


## 3. XGBoost — grid de `max_depth` x `eta`

In [ ]:
import xgboost as xgb

def hay_gpu():
    try:
        import subprocess
        subprocess.check_output(["nvidia-smi"])
        return True
    except Exception:
        return False

DEVICE_XGB = "cuda" if hay_gpu() else "cpu"
print("XGBoost device:", DEVICE_XGB)

dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_val, label=y_val)
scale_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

grid_xgb = [
    {"max_depth": 4, "eta": 0.3},
    {"max_depth": 8, "eta": 0.1},
    {"max_depth": 8, "eta": 0.3},
    {"max_depth": 12, "eta": 0.1},
]

for params in grid_xgb:
    p = {
        "objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist",
        "device": DEVICE_XGB, "subsample": 0.8, "colsample_bytree": 0.8,
        "scale_pos_weight": scale_pos_weight, "seed": SEMILLA, **params,
    }
    t0 = time.perf_counter()
    modelo = xgb.train(p, dtrain, num_boost_round=200)
    tiempo = time.perf_counter() - t0
    pred = (modelo.predict(dval) >= 0.5).astype(int)
    f1 = f1_score(y_val, pred)
    print(f"XGB {params} -> F1={f1:.4f}  ({tiempo:.1f}s)")
    resultados_busqueda.append({"modelo": "XGBoost", "config": str(params), "f1": f1, "tiempo_s": tiempo, **params})


## 4. Red Neuronal — grid de `batch_size` x `learning_rate`

Esto también responde lo que pidió Robson: cómo cambia el entrenamiento con distintos `batch_size` (parte de su análisis de escalabilidad).

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

DEVICE_NN = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Red Neuronal device:", DEVICE_NN)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train).astype(np.float32)
X_val_esc = escalador.transform(X_val).astype(np.float32)
y_train_arr, y_val_arr = y_train.values, y_val.values

def iterar_batches(X, y, batch_size, shuffle=False):
    X_t = torch.tensor(X, dtype=torch.float32)
    y_t = torch.tensor(y, dtype=torch.long)
    n = X_t.shape[0]
    idx = torch.randperm(n) if shuffle else torch.arange(n)
    for i in range(0, n, batch_size):
        sel = idx[i:i + batch_size]
        yield X_t[sel], y_t[sel]

class MLP(nn.Module):
    def __init__(self, n_entradas, n_clases=2):
        super().__init__()
        self.red = nn.Sequential(
            nn.Linear(n_entradas, 128), nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU(),
            nn.Linear(32, n_clases),
        )
    def forward(self, x):
        return self.red(x)

grid_nn = [
    {"batch_size": 2048, "lr": 1e-3},
    {"batch_size": 8192, "lr": 1e-3},
    {"batch_size": 8192, "lr": 5e-4},
    {"batch_size": 32768, "lr": 1e-3},
]
EPOCHS_BUSQUEDA = 3

for params in grid_nn:
    modelo = MLP(n_entradas=X_train_esc.shape[1]).to(DEVICE_NN)
    criterio = nn.CrossEntropyLoss()
    optimizador = torch.optim.Adam(modelo.parameters(), lr=params["lr"])

    t0 = time.perf_counter()
    for _ in range(EPOCHS_BUSQUEDA):
        modelo.train()
        for x, y in iterar_batches(X_train_esc, y_train_arr, params["batch_size"], shuffle=True):
            x, y = x.to(DEVICE_NN), y.to(DEVICE_NN)
            optimizador.zero_grad()
            perdida = criterio(modelo(x), y)
            perdida.backward()
            optimizador.step()
    tiempo = time.perf_counter() - t0

    modelo.eval()
    preds = []
    with torch.no_grad():
        for x, y in iterar_batches(X_val_esc, y_val_arr, params["batch_size"], shuffle=False):
            x = x.to(DEVICE_NN)
            preds.extend(modelo(x).argmax(1).cpu().numpy())
    f1 = f1_score(y_val_arr, np.array(preds))
    print(f"NN {params} -> F1={f1:.4f}  ({tiempo:.1f}s)")
    resultados_busqueda.append({"modelo": "Red Neuronal", "config": str(params), "f1": f1, "tiempo_s": tiempo, **params})


## 5. Tabla comparativa y mejor configuración por modelo

In [ ]:
tabla_busqueda = pd.DataFrame(resultados_busqueda)
tabla_busqueda.to_csv(os.path.join(DIR_TABLAS, "07_busqueda_hiperparametros.csv"), index=False)

# Elegir "mejor" balanceando calidad y tiempo, no solo F1:
# de las configuraciones cuyo F1 este dentro de TOLERANCIA_F1 del mejor F1 encontrado,
# se elige la MAS RAPIDA. Evita elegir una config que gana F1 por una fraccion minima
# pero tarda varias veces mas (nos paso con batch_size chico en la red neuronal:
# +0.001 F1 a cambio de 5x mas tiempo, mala relacion costo/beneficio a escala completa).
TOLERANCIA_F1 = 0.01

print("Mejor configuracion por modelo (F1 vs tiempo, con tolerancia):\n")
mejores = {}
for modelo in tabla_busqueda["modelo"].unique():
    sub = tabla_busqueda[tabla_busqueda["modelo"] == modelo].copy()
    mejor_f1_posible = sub["f1"].max()
    candidatos = sub[sub["f1"] >= mejor_f1_posible - TOLERANCIA_F1].sort_values("tiempo_s")
    elegido = candidatos.iloc[0]
    mejores[modelo] = elegido["config"]

    print(f"{modelo}:")
    print(f"  Mejor F1 posible : {mejor_f1_posible:.4f}")
    print(f"  Elegido          : {elegido['config']}  ->  F1={elegido['f1']:.4f}  ({elegido['tiempo_s']:.1f}s)")
    if elegido["f1"] < mejor_f1_posible:
        print(f"  (se sacrifico {mejor_f1_posible - elegido['f1']:.4f} de F1 a cambio de ser mas rapido)")
    print()

with open(os.path.join(DIR_TABLAS, "08_mejores_hiperparametros.json"), "w", encoding="utf-8") as f:
    json.dump(mejores, f, indent=2, ensure_ascii=False)

tabla_busqueda

In [ ]:
fig = px.bar(tabla_busqueda, x="config", y="f1", color="modelo", facet_col="modelo",
             title="F1 por configuracion probada (busqueda de hiperparametros)")
fig.update_xaxes(matches=None)
fig.show()

fig_mpl, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, modelo in zip(axes, tabla_busqueda["modelo"].unique()):
    sub = tabla_busqueda[tabla_busqueda["modelo"] == modelo]
    ax.bar(range(len(sub)), sub["f1"])
    ax.set_xticks(range(len(sub)))
    ax.set_xticklabels(sub["config"], rotation=45, ha="right", fontsize=7)
    ax.set_title(modelo)
    ax.set_ylabel("F1")
plt.tight_layout()
plt.savefig(os.path.join(DIR_FIGURAS, "09_busqueda_hiperparametros.png"), dpi=150)
plt.show()


## 6. Cómo aplicar esto al pipeline final

Toma los valores de `08_mejores_hiperparametros.json` (o los que imprimió la celda 5) y
actualiza `00_pipeline_modelado.ipynb` antes de la corrida final con `MODO="full"`:

- **Random Forest** (sección 3.1): cambia `n_estimators=300, max_depth=16` por los valores ganadores.
- **XGBoost** (sección 3.2, diccionario `params_xgb`): cambia `max_depth` y `eta`.
- **Red Neuronal** (sección 3.3): cambia `BATCH_SIZE_NN` en la rama `else` (GPU) por el valor ganador,
  y ajusta `lr=1e-3` en `torch.optim.Adam(modelo_nn.parameters(), lr=...)` si el mejor `lr` fue distinto.

Con esto, la corrida final ya no usa parámetros "por defecto" — queda documentado en el informe
que se hizo una búsqueda real (semana 12 del cronograma) y se justifica la configuración elegida.
